# BASIL-3 training and validation

Select a **GPU** runtime in Colab before running these cells. The Colab preset is a development run; `basil3.yaml` is the larger study. Both evaluate on **val** only, never test. Full training and evaluation may exceed a Colab session; checkpoints are saved to Google Drive.

In [ ]:
import os
import sys
import subprocess
from pathlib import Path
import torch

assert torch.cuda.is_available(), "Select Runtime > Change runtime type > GPU first."
print("GPU:", torch.cuda.get_device_name(0))
root = Path("/content/BASIL-2-beta")
if not root.exists():
    subprocess.run(["git", "clone", "https://github.com/saptarshihalder/BASIL-2-beta.git", str(root)], check=True)
os.chdir(root)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
subprocess.run([sys.executable, "-m", "pytest", "-q"], check=True)

## Save checkpoints

Drive is used because a Colab session can disconnect before the full run completes. Authorize the mount when Colab asks.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
base = Path("/content/drive/MyDrive/BASIL-2-runs")
base.mkdir(parents=True, exist_ok=True)

def run(*args):
    print("Running:", " ".join(map(str, args)), flush=True)
    subprocess.run([sys.executable, *map(str, args)], cwd=root, check=True)

## 1. Colab preset (seed 0)

After training, evaluate the VOC, disagreement, confidence and random gates at budget 0.25 with the same openings and seats.

In [ ]:
colab = base / "colab_s0"
run("-m", "basil3.train", "--config", "configs/basil3_colab.yaml", "--out", colab,
    "--stage", "selfplay", *(["--resume"] if (colab / "citadel.pt").exists() else []))
run("-m", "basil3.train", "--config", "configs/basil3_colab.yaml", "--out", colab, "--stage", "gate")
run("-m", "basil3.evaluate", "--run", colab, "--split", "val", "--universes", 12,
    "--openings", 2, "--sims", 40, "--device", "cuda", "--opponents", "greedy",
    "--agents", "morty,citadel@always,citadel@voc:0.25,citadel@disagreement:0.25,citadel@confidence:0.25,citadel@random:0.25",
    "--out", colab / "val_eval.json")

## 2. Full preset (seed 0)

Keep this run separate. The default configuration uses 8 search workers and may require a higher-CPU GPU runtime. If Colab interrupts an iteration, rerunning the cell resumes from the last saved checkpoint. The evaluation is costly because search is CPU-bound.

In [ ]:
full = base / "full_s0"
run("-m", "basil3.train", "--config", "configs/basil3.yaml", "--out", full,
    "--stage", "selfplay", *(["--resume"] if (full / "citadel.pt").exists() else []))
run("-m", "basil3.train", "--config", "configs/basil3.yaml", "--out", full, "--stage", "gate")
run("-m", "basil3.evaluate", "--run", full, "--split", "val", "--universes", 40,
    "--openings", 4, "--sims", 64, "--device", "cuda",
    "--opponents", "greedy,uct-100",
    "--out", full / "val_eval.json")

## Inspect the actual results

Compare VOC against the random gate using both score differences and measured imagination calls per move. These are **validation** results, not final paper results. Export the JSON and training histories before making claims.

In [ ]:
import json
for label, out in [("Colab", colab), ("Full", full)]:
    path = out / "val_eval.json"
    if not path.exists():
        continue
    obj = json.loads(path.read_text())
    print("\n", label, "seed", obj["train_seed"], "split", obj["args"]["split"])
    for r in obj["results"]:
        print(r["agent"], r["opponent"], "score", r["score"], "delta", r.get("delta_vs_morty"),
              "CI", r.get("delta_ci95"), "calls/move", r["imagination_calls_per_move"])
    print("Results:", path)
